## Import libraries
'pip install pandas scikit-learn micromlgen'

In [26]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from micromlgen import port
import shutil
import os

## Configuration parameters
Check your sheet's column name for FEATURES_COL and TARGET_COL values

In [27]:
# ========== CONFIGURATION PARAMETERS ==========
# Dataset path
DATASET_PATH = '../Dataset/DATA TRAINING.csv'

# Dataset features to use
FEATURES_COL = ['RMSSD (ms)', 'SDNN (ms)', 'BPM']
TARGET_COL = 'Label Stres'

# Label mapping for stress levels
LABEL_MAP = {'stres rendah': 0, 'stres sedang': 1, 'stres tinggi': 2}
TARGET_NAMES = list(LABEL_MAP.keys())

# Train-test split parameters
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Random Forest model parameters
RF_N_ESTIMATORS = 10
RF_MAX_DEPTH = 5

# ========== END CONFIGURATION ==========

## Load dataset

In [28]:
# 1. Muat Data
df = pd.read_csv(DATASET_PATH)

# 2. Persiapan Fitur dan Target
# RMSSD, SDNN, dan BPM
X = df[FEATURES_COL]
y = df[TARGET_COL]

In [29]:
# Encoding target ke integer
y = y.map(LABEL_MAP)

# Membagi data menjadi data latih dan uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

## Train model

In [30]:
# Preprocessing Normalisasi Data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Latih Model Random Forest
clf = RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, random_state=RANDOM_STATE)
clf.fit(X_train_scaled, y_train)

# 4. Evaluasi Model
y_pred = clf.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"Akurasi Model: {acc * 100:.2f}%")
print("\nLaporan Klasifikasi:")
print(classification_report(y_test, y_pred, target_names=TARGET_NAMES))

Akurasi Model: 98.97%

Laporan Klasifikasi:
              precision    recall  f1-score   support

stres rendah       1.00      1.00      1.00        24
stres sedang       1.00      0.98      0.99        54
stres tinggi       0.95      1.00      0.97        19

    accuracy                           0.99        97
   macro avg       0.98      0.99      0.99        97
weighted avg       0.99      0.99      0.99        97



## Export model

In [31]:
# 5. Ekspor ke Header C++
print("Mengekspor model ke model.h...")
c_code = port(clf, classname='StressPredictorRF')

scaler_code = f"""
// Normalization Constants
const float SCALER_MEANS[3] = {{{scaler.mean_[0]}f, {scaler.mean_[1]}f, {scaler.mean_[2]}f}};
const float SCALER_SCALES[3] = {{{scaler.scale_[0]}f, {scaler.scale_[1]}f, {scaler.scale_[2]}f}};

"""

c_code_combined = scaler_code + c_code

with open('model.h', 'w') as f:
    f.write(c_code_combined)

print("Berhasil! Model diekspor ke model.h")

Mengekspor model ke model.h...
Berhasil! Model diekspor ke model.h


## (Opsional) Update library include Arduino IDE

In [32]:
# Otomatis update ke library
dest_path = 'StressPredictor/src/model.h'
if os.path.exists('StressPredictor/src'):
    shutil.copy('model.h', dest_path)
    print(f"Model berhasil disalin ke {dest_path}")